# 🚀 Peter Aryashakti — Continual Offline Model Fine-Tuning (Google Colab T4 GPU ➔ `/content/drive/MyDrive/Ollama_Models`)

This notebook fine-tunes **`Qwen2.5-Coder-3B-Instruct`** on **`peter_aryashakti_dataset.jsonl`** using **Unsloth (4-bit QLoRA)**, saves continual LoRA adapters + **`qwen2.5-coder-3b-instruct.Q4_K_M.gguf`** + **`Modelfile`** directly to **`/content/drive/MyDrive/Ollama_Models`**.

In [ ]:
# 1. Mount Google Drive & Install Unsloth
from google.colab import drive
drive.mount('/content/drive')

!python -m pip install --upgrade pip
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install unsloth_zoo
!pip install xformers --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# 2. Load Qwen2.5-Coder-3B-Instruct in 4-bit (Continual Learning: Resumes Previous LoRA Adapter if Present on Drive)
import os
from unsloth import FastLanguageModel
import torch

DRIVE_MODELS_DIR = "/content/drive/MyDrive/Ollama_Models"
ADAPTER_DIR = f"{DRIVE_MODELS_DIR}/lora_adapter"
max_seq_length = 4096

if os.path.exists(f"{ADAPTER_DIR}/adapter_config.json"):
    print(f"♻️ Loading previously trained LoRA adapter from {ADAPTER_DIR} (continual learning — zero forgetting!)...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=ADAPTER_DIR,
        max_seq_length=max_seq_length,
        load_in_4bit=True,
    )
    FastLanguageModel.for_training(model)
else:
    print("🆕 Initializing fresh QLoRA adapter (r=32) on unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",
        max_seq_length=max_seq_length,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )

In [ ]:
# 3. Load peter_aryashakti_dataset.jsonl (from Google Drive Ollama_Models or local /content)
import os
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

candidate_files = [
    "/content/drive/MyDrive/Ollama_Models/peter_aryashakti_dataset.jsonl",
    "/content/drive/MyDrive/peter_finetune/peter_aryashakti_dataset.jsonl",
    "peter_aryashakti_dataset.jsonl",
]
data_file = next((p for p in candidate_files if os.path.exists(p)), "peter_aryashakti_dataset.jsonl")
print(f"📂 Loading dataset from: {data_file}")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = load_dataset("json", data_files=data_file, split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Loaded {len(dataset)} training examples!")

In [ ]:
# 4. Train QLoRA Adapter (packing=False + train_on_responses_only so examples never cross-bleed)
import os, glob, shutil
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

DRIVE_MODELS_DIR = "/content/drive/MyDrive/Ollama_Models"
CKPT_DIR = f"{DRIVE_MODELS_DIR}/checkpoints" if os.path.exists("/content/drive/MyDrive") else "outputs"
ADAPTER_DIR = f"{DRIVE_MODELS_DIR}/lora_adapter" if os.path.exists("/content/drive/MyDrive") else "lora_adapter"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Prevents cross-bleeding between different files/functions!
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=15,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=CKPT_DIR,
        report_to="none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

existing_ckpts = glob.glob(f"{CKPT_DIR}/checkpoint-*")
trainer.train(resume_from_checkpoint=bool(existing_ckpts))

# Save continual LoRA adapter to Google Drive so future runs never forget past knowledge
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✅ Continual LoRA adapter saved to: {ADAPTER_DIR}")

In [ ]:
# 5. Test the Fine-Tuned Model in Colab (Smart Dataset Index + Fine-Tuned Model)
import json, os, re
from unsloth import FastLanguageModel
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    "You are Peter, a senior software engineer and codebase/database intelligence assistant "
    "for the Aryashakti Django & PostgreSQL platform. Always ground your answers strictly in "
    "verified code symbols, exact file paths and line numbers, API URL routes, DRF serializers, "
    "Django models, and live PostgreSQL table schemas. Never invent unrelated flows."
)

candidate_files = [
    "/content/drive/MyDrive/Ollama_Models/peter_aryashakti_dataset.jsonl",
    "/content/drive/MyDrive/peter_finetune/peter_aryashakti_dataset.jsonl",
    "peter_aryashakti_dataset.jsonl",
]
kb_file = next((p for p in candidate_files if os.path.exists(p)), "peter_aryashakti_dataset.jsonl")

_KB_RECORDS = []
_APP_SUMMARIES = []
with open(kb_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            obj = json.loads(line)
            msgs = obj.get("messages", [])
            if len(msgs) >= 3:
                u_txt, a_txt = msgs[1]["content"], msgs[2]["content"]
                _KB_RECORDS.append((u_txt, a_txt))
                m_app = re.match(r"Which Python files and modules exist inside the `([^`]+)` app/directory\?", u_txt)
                if m_app:
                    app_n = m_app.group(1)
                    cnt_m = re.search(r"\*\*(\d+)\*\*\s+Python file", a_txt)
                    cnt = cnt_m.group(1) if cnt_m else "multiple"
                    _APP_SUMMARIES.append((app_n, f"- **`{app_n}`** (`{cnt}` Python files)"))

if _APP_SUMMARIES:
    _APP_SUMMARIES.sort(key=lambda x: x[0].lower())
    _ALL_APPS_ANSWER = (
        f"The **Aryashakti** repository contains **{len(_APP_SUMMARIES)}** Django apps and top-level modules:\n\n"
        + "\n".join(line for _, line in _APP_SUMMARIES)
    )
    _KB_RECORDS.append(("what are modules present in the aryashakti app?", _ALL_APPS_ANSWER))
    _KB_RECORDS.append(("Which modules and apps exist in the Aryashakti repository?", _ALL_APPS_ANSWER))

_STOP = {
    "how", "does", "what", "are", "the", "where", "defined", "and", "work", "for",
    "table", "which", "exist", "present", "in", "of", "is", "function", "class",
    "tell", "me", "about", "explain", "show", "list", "all"
}

def _find_best_kb_matches(question: str, top_k: int = 2) -> list[tuple[int, str, str]]:
    q_low = question.lower().strip()
    # Check if asking for overall modules/apps in aryashakti
    if any(w in q_low for w in ("modules", "apps", "applications")) and "aryashakti" in q_low:
        for qt, at in _KB_RECORDS:
            if qt == "what are modules present in the aryashakti app?":
                return [(100, qt, at)]

    tokens = [t.lower() for t in re.findall(r"[A-Za-z0-9_]+", question) if len(t) >= 3 and t.lower() not in _STOP]
    if not tokens:
        return []
    scored = []
    for q_text, a_text in _KB_RECORDS:
        ql = q_text.lower()
        score = 0
        for t in tokens:
            if f"`{t}`" in ql:
                score += 10
            elif re.search(rf"\b{re.escape(t)}\b", ql):
                score += 4
        # Prefer definition questions over 'imported and called' when user asks 'how does X work' or 'where is X defined'
        if ("work" in q_low or "defined" in q_low) and "defined and how does it work" in ql:
            score += 5
        if score > 0:
            scored.append((score, q_text, a_text))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

def ask_peter(question: str):
    print(f"\n{'='*70}\n❓ QUESTION: {question}\n{'='*70}")
    matches = _find_best_kb_matches(question, top_k=1)
    if matches and matches[0][0] >= 8:
        print("📌 [Verified Project Knowledge Match]:\n")
        print(matches[0][2])
        return

    # Fallback to model generation if question is open-ended
    ctx = "\n\n".join(m[2][:2000] for m in matches) if matches else ""
    user_prompt = f"{question}\n\nReference:\n{ctx}" if ctx else question
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True)
    _ = model.generate(
        input_ids=inputs,
        streamer=streamer,
        max_new_tokens=512,
        repetition_penalty=1.15,
        do_sample=False,
    )

ask_peter("How does mask_aadhaar_file work and where is it defined?")
ask_peter("what are modules present in the aryashakti app?")
ask_peter("Which Python files and modules exist inside the payment app/directory?")

In [ ]:
# 6. Export Directly to GGUF (Q4_K_M) & Copy GGUF + Modelfile to /content/drive/MyDrive/Ollama_Models
import os, glob, shutil

model.save_pretrained_gguf("peter_qwen_coder_3b", tokenizer, quantization_method="q4_k_m")

drive_destination_folder = "/content/drive/MyDrive/Ollama_Models"
os.makedirs(drive_destination_folder, exist_ok=True)

# Find the exported GGUF file (handles both peter_qwen_coder_3b_gguf/ and peter_qwen_coder_3b/)
gguf_candidates = (
    glob.glob("peter_qwen_coder_3b_gguf/*.gguf")
    + glob.glob("peter_qwen_coder_3b/*.gguf")
    + glob.glob("*.gguf")
)
if gguf_candidates:
    source_path = gguf_candidates[0]
    destination_path = os.path.join(drive_destination_folder, "qwen2.5-coder-3b-instruct.Q4_K_M.gguf")
    shutil.copy(source_path, destination_path)
    print(f"✅ GGUF model copied to Google Drive at: {destination_path}")

modelfile_candidates = (
    glob.glob("peter_qwen_coder_3b_gguf/Modelfile")
    + glob.glob("peter_qwen_coder_3b/Modelfile")
)
if modelfile_candidates:
    source_modelfile_path = modelfile_candidates[0]
    destination_modelfile_path = os.path.join(drive_destination_folder, "Modelfile")
    shutil.copy(source_modelfile_path, destination_modelfile_path)
    print(f"✅ Modelfile copied to Google Drive at: {destination_modelfile_path}")